# Self-Attention

## 注意力机制

注意力机制的核心思想是：根据 Query 与 Key 的匹配程度，计算不同信息的重要性，再对 Value 进行加权求和。

- Q（Query）：我想寻找什么信息？
- K（Key）：我具有什么特征，可以被怎样的查询匹配？
- V（Value）：如果关注到我，我能提供什么信息？

首先，通过可学习的投影矩阵得到 Q、K、V：
```
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
```

然后计算注意力分数，经 Softmax 转化为注意力权重，最后对 V 加权求和：

```
S = Q @ K.T / sqrt(d_k)
A = softmax(S)（按行计算）
H = A @ V
```

其中，A 是注意力权重矩阵，H 是 Attention 的最终输出。

## Self-Attention（自注意力）

自注意力是注意力机制的一种形式，其特点是 Q、K、V 都来自同一组输入 Token，但分别经过不同的投影矩阵变换。

通过 Self-Attention，每个 Token 都能够根据与其他 Token 的匹配程度，汇总整个序列中的相关信息，从而得到融合上下文的新特征表示。

## Self-Attention 实践

### 编码

In [1]:
import numpy as np

In [2]:
X = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 0]])
X.shape

(4, 3)

In [3]:
# X.shape=(4, 3), 希望Q, K, V的形状是(4, 2)
W_Q = np.array([
    [1, 2],
    [0, 1],
    [2, 0]])
W_K = np.array([
    [3, 1],
    [-1, 2],
    [0, -1]])
W_V = np.array([
    [1, -1],
    [2, 0],
    [0, 1]])
W_Q.shape, W_K.shape, W_V.shape

((3, 2), (3, 2), (3, 2))

In [4]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
Q, K, V

(array([[1, 2],
        [0, 1],
        [2, 0],
        [1, 3]]),
 array([[ 3,  1],
        [-1,  2],
        [ 0, -1],
        [ 2,  3]]),
 array([[ 1, -1],
        [ 2,  0],
        [ 0,  1],
        [ 3, -1]]))

缩放点积模型(Scaled Dot-Product Attention)：
$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

最终$S$的形状是$4\times4$（本例）， 其中元素 $S_{ij}$表示第 $i$ 个 Token 查询第 $j$ 个 Token 时的缩放后匹配分数。

In [5]:
S = Q @ K.T / np.sqrt(2)
S, S.shape

(array([[ 3.53553391,  2.12132034, -1.41421356,  5.65685425],
        [ 0.70710678,  1.41421356, -0.70710678,  2.12132034],
        [ 4.24264069, -1.41421356,  0.        ,  2.82842712],
        [ 4.24264069,  3.53553391, -2.12132034,  7.77817459]]),
 (4, 4))

使用 Softmax 计算注意力权重
公式为：
$$
A_{ij}=\frac{e^{S_{ij}}}{\sum_{k=1}^{4}e^{S_{ik}}}
$$

注意：Softmax 要对每一行分别计算，因为每个 Token 都要独立决定如何分配自己的注意力。

为了数值稳定（防止计算指数 $e^x$ 时发生数值溢出），我们通常先减去每一行的最大值：
$$
S'=S-\max(S,\text{axis}=1)
$$

再计算 Softmax。这不会改变最终的注意力权重。

In [6]:
# S每一行减去这一行的最大值
S_rmax = S.max(axis=1, keepdims=True)
S_shifted = S - S_rmax
S_shifted

array([[-2.12132034, -3.53553391, -7.07106781,  0.        ],
       [-1.41421356, -0.70710678, -2.82842712,  0.        ],
       [ 0.        , -5.65685425, -4.24264069, -1.41421356],
       [-3.53553391, -4.24264069, -9.89949494,  0.        ]])

In [7]:
# 定义softmax
def softmax(S):
    S_exp = np.exp(S)
    sum_row = np.sum(S_exp, axis=1, keepdims=True)
    return S_exp / sum_row

In [8]:
A = softmax(S_shifted)
A, A.sum(axis=1)

(array([[1.04249777e-01, 2.53448654e-02, 7.38630306e-04, 8.69666727e-01],
        [1.35419111e-01, 2.74645528e-01, 3.29226520e-02, 5.57012709e-01],
        [7.93034103e-01, 2.77045613e-03, 1.13955797e-02, 1.92799861e-01],
        [2.79266258e-02, 1.37697448e-02, 4.81044560e-05, 9.58255525e-01]]),
 array([1., 1., 1., 1.]))

In [9]:
# 最终结果
H = A @ V
H

array([[ 2.76393969, -0.97317787],
       [ 2.35574829, -0.65950917],
       [ 1.3769746 , -0.97443838],
       [ 2.93023269, -0.98613405]])

### 结果分析

注意力权重矩阵$A$的第$i$行代表第$i$个 token 对所有 token （包括自己）的注意力权重。

以第一行为例，第一个 token 对第四个 token 分配了$86.9\%$的权重，因此最终结果$H$的第一行最接近 token4 的$V=[3, -1]$，说明 token1 的输出主要融合了 token4 的信息。

- H 的第一列是汇总后的第一维特征。
- H 的第二列是汇总后的第二维特征。
- 具体数值由 Value 和注意力权重共同决定。

## 探究Q、K、V对结果的影响

### W的影响

In [10]:
# 让W_V每个元素变为原来的2倍
W_V2 = W_V * 2
V2 = X @ W_V2
H2 = A @ V2
np.allclose(H2, H * 2)

True

### Q的影响

保持 K、V 不变，只把第一个 Token 的 Query 放大 2 倍。

In [11]:
Q2 = Q.copy()
Q2[0] *= 2
Q2

array([[2, 4],
       [0, 1],
       [2, 0],
       [1, 3]])

In [12]:
S2 = Q2 @ K.T / np.sqrt(2)
A2 = softmax(S2)
H2 = A2 @ V
A2, A, H2, H

(array([[1.41541746e-02, 8.36593057e-04, 7.10539988e-07, 9.85008522e-01],
        [1.35419111e-01, 2.74645528e-01, 3.29226520e-02, 5.57012709e-01],
        [7.93034103e-01, 2.77045613e-03, 1.13955797e-02, 1.92799861e-01],
        [2.79266258e-02, 1.37697448e-02, 4.81044560e-05, 9.58255525e-01]]),
 array([[1.04249777e-01, 2.53448654e-02, 7.38630306e-04, 8.69666727e-01],
        [1.35419111e-01, 2.74645528e-01, 3.29226520e-02, 5.57012709e-01],
        [7.93034103e-01, 2.77045613e-03, 1.13955797e-02, 1.92799861e-01],
        [2.79266258e-02, 1.37697448e-02, 4.81044560e-05, 9.58255525e-01]]),
 array([[ 2.97085293, -0.99916199],
        [ 2.35574829, -0.65950917],
        [ 1.3769746 , -0.97443838],
        [ 2.93023269, -0.98613405]]),
 array([[ 2.76393969, -0.97317787],
        [ 2.35574829, -0.65950917],
        [ 1.3769746 , -0.97443838],
        [ 2.93023269, -0.98613405]]))

解释：

对$Q$的第一行乘以2之后，该向量的方向没有变，但是长度变化为原来的两倍。

当 Query 乘以大于 1 的正数时，其方向保持不变，各 Key 的匹配分数之间的差距被放大，Softmax 的输出通常会更加集中于原本匹配分数最高的 Token。在本实验中，Token 1 更加集中地关注 Token 4，因此输出 H 更接近 Token 4 的 Value。

即以前关注的对象，现在会更加关注；这也与实验结果相符，以前的H1最接近V4，改变Q后更接近了。

如果改变了Q1的方向，那么token1关注的方向就会不同，可能最终结果（加权和）会更接近其他token。

### K的影响

保持原始 Q、V 不变，只修改第四个 Token 的 Key

In [13]:
K2 = K.copy()
K2[3] *= 2
K2

array([[ 3,  1],
       [-1,  2],
       [ 0, -1],
       [ 4,  6]])

In [14]:
S3 = Q @ K2.T / np.sqrt(2)
A3 = softmax(S3)
H3 = A3 @ V
A3, A, H3, H

(array([[4.18556777e-04, 1.01758157e-04, 2.96555761e-06, 9.99476720e-01],
        [2.66066692e-02, 5.39613845e-02, 6.46852654e-03, 9.12963420e-01],
        [1.94889473e-01, 6.80844285e-04, 2.80048301e-03, 8.01629199e-01],
        [1.22042447e-05, 6.01753096e-06, 2.10221799e-08, 9.99981757e-01]]),
 array([[1.04249777e-01, 2.53448654e-02, 7.38630306e-04, 8.69666727e-01],
        [1.35419111e-01, 2.74645528e-01, 3.29226520e-02, 5.57012709e-01],
        [7.93034103e-01, 2.77045613e-03, 1.13955797e-02, 1.92799861e-01],
        [2.79266258e-02, 1.37697448e-02, 4.81044560e-05, 9.58255525e-01]]),
 array([[ 2.99905223, -0.99989231],
        [ 2.8734197 , -0.93310156],
        [ 2.60113876, -0.99371819],
        [ 2.99996951, -0.99999394]]),
 array([[ 2.76393969, -0.97317787],
        [ 2.35574829, -0.65950917],
        [ 1.3769746 , -0.97443838],
        [ 2.93023269, -0.98613405]]))

修改一个 Token 的 Key，会改变所有 Query 与这个 Key 的匹配分数，但不同 Query 的注意力权重可能朝不同方向变化。

如果本来注意力分数是正的，那么某个key变大会导致该注意力分数增加，间接导致权重增加，也就是对应token更加关注本token。反之，注意力分数本来是负的，会导致权重进一步减小，也就是更不关注。

当某个 Key 被乘以大于 1 的正数时，如果它与某个 Query 的原始点积为正，那么匹配分数会增加，对应的注意力权重也会增大；如果原始点积为负，那么匹配分数会降低，对应的注意力权重也会减小。这里假设其他 Key 不变。